# Generate the Annual Stochastic Event Catalog

This notebook generates a reproducible annual earthquake event catalog from
the rupture-level annual occurrence rates extracted in Notebook 2.

The catalog includes:

- Cascadia interface earthquakes
- Oregon intraslab earthquakes
- the USGS source-tree and rupture-set metadata associated with each event
- a simulated calendar year and time within the year
- reproducible event identifiers and random seeds

The catalog represents the branch-weighted mean USGS model. Each rupture is
treated as an independent Poisson occurrence process using its validated
weighted annual rate.

The same event catalog will be used throughout the remaining workflow. The
independent and spatially correlated ground-motion cases must not use
different earthquake occurrences.

A two-million-year simulation horizon is used for the internship-ready
baseline. Empty years are represented implicitly rather than stored as
individual rows.

In [1]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if candidate.name == "seismic-correlation-insurance-loss":
            return candidate

    raise RuntimeError(
        "Could not locate the seismic-correlation-insurance-loss repository."
    )


def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as file_object:
        for block in iter(
            lambda: file_object.read(1024 * 1024),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


def boolean_series(values):
    if pd.api.types.is_bool_dtype(values):
        return values

    return (
        values.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
            }
        )
    )


PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
METADATA_DIR = DATA_DIR / "metadata"

RUPTURE_RATE_DIR = (
    DATA_DIR
    / "processed"
    / "usgs_rupture_rates"
)

CATALOG_DIR = (
    DATA_DIR
    / "processed"
    / "annual_event_catalog"
)

CATALOG_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


CASCADIA_RATE_PATH = (
    RUPTURE_RATE_DIR
    / "cascadia_interface_rupture_rates.csv"
)

OREGON_RATE_PATH = (
    RUPTURE_RATE_DIR
    / "oregon_intraslab_rupture_rates.csv"
)

NOTEBOOK_2_VALIDATION_PATH = (
    METADATA_DIR
    / "notebook_2_final_validation.csv"
)

NOTEBOOK_2_METADATA_PATH = (
    METADATA_DIR
    / "notebook_2_completion_metadata.json"
)

CATALOG_CONFIG_PATH = (
    METADATA_DIR
    / "annual_event_catalog_config.json"
)

EVENT_CATALOG_PATH = (
    CATALOG_DIR
    / "annual_event_catalog.csv"
)

ANNUAL_COUNT_PATH = (
    CATALOG_DIR
    / "annual_event_counts.csv"
)


required_files = {
    "Cascadia rupture rates": CASCADIA_RATE_PATH,
    "Oregon rupture rates": OREGON_RATE_PATH,
    "Notebook 2 validation": NOTEBOOK_2_VALIDATION_PATH,
    "Notebook 2 completion metadata": NOTEBOOK_2_METADATA_PATH,
}


file_validation = pd.DataFrame(
    [
        {
            "file": description,
            "path": str(path),
            "exists": path.is_file(),
            "size_mb": (
                path.stat().st_size / 1024**2
                if path.is_file()
                else np.nan
            ),
        }
        for description, path in required_files.items()
    ]
)


print("Notebook 2 inputs:")
display(file_validation)


if not file_validation["exists"].all():
    missing = file_validation.loc[
        ~file_validation["exists"],
        "path",
    ].tolist()

    raise FileNotFoundError(
        "Required Notebook 2 files are missing:\n"
        + "\n".join(
            f"  {path}"
            for path in missing
        )
    )


notebook_2_validation = pd.read_csv(
    NOTEBOOK_2_VALIDATION_PATH
)

notebook_2_validation["blocking"] = boolean_series(
    notebook_2_validation["blocking"]
)

notebook_2_validation["passes"] = boolean_series(
    notebook_2_validation["passes"]
)


blocking_failures = notebook_2_validation.loc[
    notebook_2_validation["blocking"]
    & ~notebook_2_validation["passes"],
    "check",
].tolist()


if blocking_failures:
    raise RuntimeError(
        "Notebook 2 contains unresolved blocking validation failures:\n"
        + "\n".join(
            f"  {check}"
            for check in blocking_failures
        )
    )


notebook_2_metadata = json.loads(
    NOTEBOOK_2_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)


cascadia = pd.read_csv(
    CASCADIA_RATE_PATH,
    low_memory=False,
)

oregon = pd.read_csv(
    OREGON_RATE_PATH,
    low_memory=False,
)


rupture_rates = pd.concat(
    [
        cascadia,
        oregon,
    ],
    ignore_index=True,
)


numeric_columns = [
    "magnitude",
    "rake_deg",
    "branch_weight",
    "rupture_set_weight",
    "raw_annual_rate",
    "weighted_annual_rate",
]

for column in numeric_columns:
    rupture_rates[column] = pd.to_numeric(
        rupture_rates[column],
        errors="coerce",
    )


expected_groups = {
    "cascadia_interface",
    "oregon_intraslab",
}

actual_groups = set(
    rupture_rates[
        "target_group"
    ].dropna()
)


rate_formula_matches = np.allclose(
    rupture_rates["weighted_annual_rate"],
    (
        rupture_rates["raw_annual_rate"]
        * rupture_rates["rupture_set_weight"]
    ),
    rtol=1e-12,
    atol=1e-18,
    equal_nan=False,
)


actual_cascadia_hash = file_sha256(
    CASCADIA_RATE_PATH
)

actual_oregon_hash = file_sha256(
    OREGON_RATE_PATH
)

expected_cascadia_hash = notebook_2_metadata.get(
    "cascadia_export_sha256"
)

expected_oregon_hash = notebook_2_metadata.get(
    "oregon_export_sha256"
)


total_annual_rate = float(
    rupture_rates[
        "weighted_annual_rate"
    ].sum()
)


metadata_total_rate = float(
    notebook_2_metadata[
        "total_weighted_annual_rate"
    ]
)


CATALOG_YEARS = 2_000_000
MASTER_SEED = 20260729

expected_event_count = (
    total_annual_rate
    * CATALOG_YEARS
)

expected_nonempty_years = (
    CATALOG_YEARS
    * (
        1.0
        - np.exp(
            -total_annual_rate
        )
    )
)

expected_multi_event_years = (
    CATALOG_YEARS
    * (
        1.0
        - np.exp(
            -total_annual_rate
        )
        * (
            1.0
            + total_annual_rate
        )
    )
)


return_periods = pd.DataFrame(
    {
        "return_period_years": [
            225,
            475,
            975,
            2475,
        ]
    }
)

return_periods[
    "expected_annual_observations_above_level"
] = (
    CATALOG_YEARS
    / return_periods[
        "return_period_years"
    ]
)


source_summary = (
    rupture_rates
    .groupby(
        "target_group",
        dropna=False,
    )
    .agg(
        rupture_rows=(
            "rupture_id",
            "size",
        ),
        minimum_magnitude=(
            "magnitude",
            "min",
        ),
        maximum_magnitude=(
            "magnitude",
            "max",
        ),
        weighted_annual_rate=(
            "weighted_annual_rate",
            "sum",
        ),
    )
    .reset_index()
)


validation = pd.DataFrame(
    [
        {
            "check": "Notebook 2 blocking checks passed",
            "passes": len(blocking_failures) == 0,
        },
        {
            "check": "Both source groups are present",
            "passes": actual_groups == expected_groups,
        },
        {
            "check": "Expected rupture-row count retained",
            "passes": len(rupture_rates) == 23416,
        },
        {
            "check": "Cascadia rupture-row count retained",
            "passes": len(cascadia) == 11101,
        },
        {
            "check": "Oregon rupture-row count retained",
            "passes": len(oregon) == 12315,
        },
        {
            "check": "Rupture identifiers are unique",
            "passes": (
                rupture_rates[
                    "rupture_id"
                ].duplicated().sum()
                == 0
            ),
        },
        {
            "check": "All weighted rates are finite",
            "passes": np.isfinite(
                rupture_rates[
                    "weighted_annual_rate"
                ]
            ).all(),
        },
        {
            "check": "All weighted rates are positive",
            "passes": (
                rupture_rates[
                    "weighted_annual_rate"
                ]
                > 0
            ).all(),
        },
        {
            "check": "Weighted-rate formula is preserved",
            "passes": rate_formula_matches,
        },
        {
            "check": "Cascadia checksum matches Notebook 2",
            "passes": (
                actual_cascadia_hash
                == expected_cascadia_hash
            ),
        },
        {
            "check": "Oregon checksum matches Notebook 2",
            "passes": (
                actual_oregon_hash
                == expected_oregon_hash
            ),
        },
        {
            "check": "Total annual rate matches Notebook 2",
            "passes": bool(
                np.isclose(
                    total_annual_rate,
                    metadata_total_rate,
                    rtol=1e-12,
                    atol=1e-15,
                )
            ),
        },
        {
            "check": (
                "Catalog provides at least 500 observations "
                "at the 2475-year probability level"
            ),
            "passes": (
                CATALOG_YEARS
                / 2475
                >= 500
            ),
        },
    ]
)


print("\nRupture-rate summary:")
display(source_summary)

print("\nReturn-period sampling depth:")
display(return_periods)

print("\nNotebook 3 input validation:")
display(validation)


if not validation["passes"].all():
    failed_checks = validation.loc[
        ~validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Notebook 3 input validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed_checks
        )
    )


catalog_config = {
    "notebook": "03_generate_annual_event_catalog.ipynb",
    "catalog_name": "baseline_mean_rate_catalog_2m_years",
    "catalog_years": CATALOG_YEARS,
    "master_seed": MASTER_SEED,
    "occurrence_model": (
        "Independent Poisson occurrence counts for each rupture "
        "using its branch-weighted annual rate"
    ),
    "calendar_assignment": (
        "Uniform discrete year and uniform time within each year"
    ),
    "zero_year_storage": (
        "Empty years are represented implicitly"
    ),
    "common_catalog_rule": (
        "The same event catalog must be reused for all later "
        "ground-motion dependence cases"
    ),
    "rupture_rows": int(
        len(rupture_rates)
    ),
    "total_annual_rate": total_annual_rate,
    "expected_event_count": expected_event_count,
    "expected_nonempty_years": expected_nonempty_years,
    "expected_multi_event_years": expected_multi_event_years,
    "cascadia_rate_file": str(
        CASCADIA_RATE_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "cascadia_rate_sha256": actual_cascadia_hash,
    "oregon_rate_file": str(
        OREGON_RATE_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "oregon_rate_sha256": actual_oregon_hash,
    "event_catalog_output": str(
        EVENT_CATALOG_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "annual_count_output": str(
        ANNUAL_COUNT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


CATALOG_CONFIG_PATH.write_text(
    json.dumps(
        catalog_config,
        indent=2,
    ),
    encoding="utf-8",
)


print("\nNOTEBOOK 3 INPUT VALIDATION PASSED")

print(f"\nCatalog years:             {CATALOG_YEARS:,}")
print(f"Master seed:               {MASTER_SEED}")
print(f"Rupture rows:              {len(rupture_rates):,}")
print(f"Total annual rate:         {total_annual_rate:.14g}")
print(f"Expected earthquake count: {expected_event_count:,.2f}")
print(f"Expected nonempty years:   {expected_nonempty_years:,.2f}")
print(f"Expected multi-event years:{expected_multi_event_years:,.2f}")
print(f"\nConfiguration:\n  {CATALOG_CONFIG_PATH}")

print(
    "\nNext step: sample the Poisson occurrence counts "
    "and assign each event to a simulated year."
)

Notebook 2 inputs:


,file,path,exists,size_mb
0,Cascadia rupture rates,C:\Users\USER\Documents\GitHub\seismic-correla...,True,5.690289
1,Oregon rupture rates,C:\Users\USER\Documents\GitHub\seismic-correla...,True,5.186740
2,Notebook 2 validation,C:\Users\USER\Documents\GitHub\seismic-correla...,True,0.000876
3,Notebook 2 completion metadata,C:\Users\USER\Documents\GitHub\seismic-correla...,True,0.001462



Rupture-rate summary:


,target_group,rupture_rows,minimum_magnitude,maximum_magnitude,weighted_annual_rate
0,cascadia_interface,11101,8.00,9.34,0.003307
1,oregon_intraslab,12315,6.55,7.95,0.001916



Return-period sampling depth:


,return_period_years,expected_annual_observations_above_level
0,225,8888.888889
1,475,4210.526316
2,975,2051.282051
3,2475,808.080808



Notebook 3 input validation:


,check,passes
0,Notebook 2 blocking checks passed,True
1,Both source groups are present,True
2,Expected rupture-row count retained,True
3,Cascadia rupture-row count retained,True
4,Oregon rupture-row count retained,True
5,Rupture identifiers are unique,True
6,All weighted rates are finite,True
7,All weighted rates are positive,True
8,Weighted-rate formula is preserved,True
9,Cascadia checksum matches Notebook 2,True



NOTEBOOK 3 INPUT VALIDATION PASSED

Catalog years:             2,000,000
Master seed:               20260729
Rupture rows:              23,416
Total annual rate:         0.0052232214574573
Expected earthquake count: 10,446.44
Expected nonempty years:   10,419.21
Expected multi-event years:27.19

Configuration:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\annual_event_catalog_config.json

Next step: sample the Poisson occurrence counts and assign each event to a simulated year.


## Generate the stochastic event catalog

For each rupture, the number of occurrences over the simulation horizon is
sampled from a Poisson distribution using its branch-weighted annual rate.

Conditional on the sampled occurrence count, event times are distributed
uniformly over the two-million-year catalog period.

The output stores only years containing earthquakes. Empty years remain
implicit.

Each event receives reproducible random seeds that will be reused in later
ground-motion calculations. In particular, the independent and spatially
correlated cases will use the same event occurrences and the same underlying
random-number streams.

In [2]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display


RUPTURE_COUNT_PATH = (
    CATALOG_DIR
    / "rupture_occurrence_counts.csv"
)

CATALOG_SUMMARY_PATH = (
    METADATA_DIR
    / "annual_event_catalog_summary.csv"
)

CATALOG_VALIDATION_PATH = (
    METADATA_DIR
    / "annual_event_catalog_validation.csv"
)

CATALOG_METADATA_PATH = (
    METADATA_DIR
    / "annual_event_catalog_metadata.json"
)


required_columns = [
    "rupture_id",
    "target_group",
    "weighted_annual_rate",
]

missing_columns = [
    column
    for column in required_columns
    if column not in rupture_rates.columns
]

if missing_columns:
    raise RuntimeError(
        "Required rupture-rate columns are missing:\n"
        + "\n".join(
            f"  {column}"
            for column in missing_columns
        )
    )


rates = rupture_rates[
    "weighted_annual_rate"
].to_numpy(
    dtype=float
)


if not np.isfinite(rates).all():
    raise RuntimeError(
        "One or more weighted annual rates are not finite."
    )

if not (rates > 0).all():
    raise RuntimeError(
        "All weighted annual rates must be positive."
    )


seed_sequence = np.random.SeedSequence(
    MASTER_SEED
)

count_seed, year_seed, time_seed = (
    seed_sequence.spawn(3)
)

count_rng = np.random.Generator(
    np.random.PCG64DXSM(
        count_seed
    )
)

year_rng = np.random.Generator(
    np.random.PCG64DXSM(
        year_seed
    )
)

time_rng = np.random.Generator(
    np.random.PCG64DXSM(
        time_seed
    )
)


expected_counts = (
    rates
    * CATALOG_YEARS
)

sampled_counts = count_rng.poisson(
    expected_counts
).astype(
    np.int64
)


total_events = int(
    sampled_counts.sum()
)

if total_events == 0:
    raise RuntimeError(
        "The simulated catalog contains no earthquakes."
    )


rupture_count_table = rupture_rates[
    [
        "rupture_id",
        "target_group",
        "magnitude",
        "weighted_annual_rate",
    ]
].copy()

rupture_count_table[
    "expected_occurrences"
] = expected_counts

rupture_count_table[
    "sampled_occurrences"
] = sampled_counts

rupture_count_table[
    "occurrence_difference"
] = (
    rupture_count_table[
        "sampled_occurrences"
    ]
    - rupture_count_table[
        "expected_occurrences"
    ]
)

rupture_count_table.to_csv(
    RUPTURE_COUNT_PATH,
    index=False,
)


rupture_row_indices = np.repeat(
    np.arange(
        len(rupture_rates),
        dtype=np.int64,
    ),
    sampled_counts,
)


occurrence_numbers = np.concatenate(
    [
        np.arange(
            1,
            count + 1,
            dtype=np.int64,
        )
        for count in sampled_counts
        if count > 0
    ]
)


if len(rupture_row_indices) != total_events:
    raise RuntimeError(
        "Expanded rupture indices do not match the sampled event count."
    )

if len(occurrence_numbers) != total_events:
    raise RuntimeError(
        "Occurrence numbers do not match the sampled event count."
    )


catalog = rupture_rates.iloc[
    rupture_row_indices
].copy()

catalog.reset_index(
    drop=True,
    inplace=True,
)

catalog.insert(
    0,
    "rupture_occurrence_number",
    occurrence_numbers,
)


catalog.insert(
    0,
    "simulation_year",
    year_rng.integers(
        low=1,
        high=CATALOG_YEARS + 1,
        size=total_events,
        dtype=np.int64,
    ),
)


catalog.insert(
    1,
    "time_within_year",
    time_rng.random(
        total_events
    ),
)


catalog.insert(
    2,
    "continuous_event_time",
    (
        catalog[
            "simulation_year"
        ].to_numpy(
            dtype=float
        )
        - 1.0
        + catalog[
            "time_within_year"
        ].to_numpy(
            dtype=float
        )
    ),
)


def deterministic_seed(
    label,
    rupture_id,
    occurrence_number,
):
    seed_text = (
        f"{MASTER_SEED}|"
        f"{label}|"
        f"{rupture_id}|"
        f"{occurrence_number}"
    )

    digest = hashlib.sha256(
        seed_text.encode(
            "utf-8"
        )
    ).digest()

    seed = int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False,
    )

    return seed & (
        (1 << 63) - 1
    )


seed_inputs = zip(
    catalog[
        "rupture_id"
    ].astype(str),
    catalog[
        "rupture_occurrence_number"
    ].astype(int),
)


seed_inputs = list(
    seed_inputs
)


catalog[
    "event_seed"
] = [
    deterministic_seed(
        "event",
        rupture_id,
        occurrence_number,
    )
    for rupture_id, occurrence_number
    in seed_inputs
]


catalog[
    "between_event_seed"
] = [
    deterministic_seed(
        "between_event",
        rupture_id,
        occurrence_number,
    )
    for rupture_id, occurrence_number
    in seed_inputs
]


catalog[
    "within_event_seed"
] = [
    deterministic_seed(
        "within_event",
        rupture_id,
        occurrence_number,
    )
    for rupture_id, occurrence_number
    in seed_inputs
]


catalog.sort_values(
    [
        "simulation_year",
        "time_within_year",
        "rupture_id",
        "rupture_occurrence_number",
    ],
    kind="stable",
    inplace=True,
)

catalog.reset_index(
    drop=True,
    inplace=True,
)


catalog.insert(
    0,
    "event_id",
    [
        f"EVT_{event_number:08d}"
        for event_number in range(
            1,
            total_events + 1,
        )
    ],
)


catalog[
    "event_number_in_year"
] = (
    catalog
    .groupby(
        "simulation_year",
        sort=False,
    )
    .cumcount()
    + 1
)


events_per_year = (
    catalog
    .groupby(
        "simulation_year",
        sort=True,
    )
    .size()
    .rename(
        "event_count"
    )
    .reset_index()
)


catalog = catalog.merge(
    events_per_year,
    on="simulation_year",
    how="left",
    validate="many_to_one",
)


catalog.rename(
    columns={
        "event_count": "events_in_year",
    },
    inplace=True,
)


events_per_year[
    "is_multi_event_year"
] = (
    events_per_year[
        "event_count"
    ]
    >= 2
)


events_per_year.to_csv(
    ANNUAL_COUNT_PATH,
    index=False,
)


event_columns = [
    "event_id",
    "simulation_year",
    "time_within_year",
    "continuous_event_time",
    "event_number_in_year",
    "events_in_year",
    "rupture_occurrence_number",
    "event_seed",
    "between_event_seed",
    "within_event_seed",
]


remaining_columns = [
    column
    for column in catalog.columns
    if column not in event_columns
]


catalog = catalog[
    [
        *event_columns,
        *remaining_columns,
    ]
]


catalog.to_csv(
    EVENT_CATALOG_PATH,
    index=False,
)


group_realizations = (
    catalog
    .groupby(
        "target_group",
        dropna=False,
    )
    .agg(
        realized_events=(
            "event_id",
            "size",
        ),
        nonempty_years=(
            "simulation_year",
            "nunique",
        ),
        minimum_magnitude=(
            "magnitude",
            "min",
        ),
        maximum_magnitude=(
            "magnitude",
            "max",
        ),
    )
    .reset_index()
)


group_rates = (
    rupture_rates
    .groupby(
        "target_group",
        dropna=False,
    )[
        "weighted_annual_rate"
    ]
    .sum()
    .rename(
        "annual_rate"
    )
    .reset_index()
)


group_summary = group_rates.merge(
    group_realizations,
    on="target_group",
    how="outer",
    validate="one_to_one",
)


group_summary[
    "expected_events"
] = (
    group_summary[
        "annual_rate"
    ]
    * CATALOG_YEARS
)


group_summary[
    "poisson_standard_deviation"
] = np.sqrt(
    group_summary[
        "expected_events"
    ]
)


group_summary[
    "event_count_z_score"
] = (
    group_summary[
        "realized_events"
    ]
    - group_summary[
        "expected_events"
    ]
) / group_summary[
    "poisson_standard_deviation"
]


observed_nonempty_years = int(
    len(
        events_per_year
    )
)

observed_multi_event_years = int(
    events_per_year[
        "is_multi_event_year"
    ].sum()
)

observed_empty_years = (
    CATALOG_YEARS
    - observed_nonempty_years
)


expected_total_events = (
    total_annual_rate
    * CATALOG_YEARS
)

total_event_standard_deviation = np.sqrt(
    expected_total_events
)

total_event_z_score = (
    total_events
    - expected_total_events
) / total_event_standard_deviation


nonempty_probability = (
    1.0
    - np.exp(
        -total_annual_rate
    )
)

multi_event_probability = (
    1.0
    - np.exp(
        -total_annual_rate
    )
    * (
        1.0
        + total_annual_rate
    )
)


expected_nonempty_years = (
    CATALOG_YEARS
    * nonempty_probability
)

expected_multi_event_years = (
    CATALOG_YEARS
    * multi_event_probability
)


nonempty_year_standard_deviation = np.sqrt(
    CATALOG_YEARS
    * nonempty_probability
    * (
        1.0
        - nonempty_probability
    )
)


multi_event_year_standard_deviation = np.sqrt(
    CATALOG_YEARS
    * multi_event_probability
    * (
        1.0
        - multi_event_probability
    )
)


nonempty_year_z_score = (
    observed_nonempty_years
    - expected_nonempty_years
) / nonempty_year_standard_deviation


multi_event_year_z_score = (
    observed_multi_event_years
    - expected_multi_event_years
) / multi_event_year_standard_deviation


summary = pd.DataFrame(
    [
        {
            "metric": "Total earthquake occurrences",
            "expected": expected_total_events,
            "realized": total_events,
            "standard_deviation": (
                total_event_standard_deviation
            ),
            "z_score": total_event_z_score,
        },
        {
            "metric": "Nonempty catalog years",
            "expected": expected_nonempty_years,
            "realized": observed_nonempty_years,
            "standard_deviation": (
                nonempty_year_standard_deviation
            ),
            "z_score": nonempty_year_z_score,
        },
        {
            "metric": "Multi-event catalog years",
            "expected": expected_multi_event_years,
            "realized": observed_multi_event_years,
            "standard_deviation": (
                multi_event_year_standard_deviation
            ),
            "z_score": multi_event_year_z_score,
        },
    ]
)


summary.to_csv(
    CATALOG_SUMMARY_PATH,
    index=False,
)


event_ids_unique = (
    catalog[
        "event_id"
    ].duplicated().sum()
    == 0
)


event_seeds_unique = (
    catalog[
        "event_seed"
    ].duplicated().sum()
    == 0
)


between_event_seeds_unique = (
    catalog[
        "between_event_seed"
    ].duplicated().sum()
    == 0
)


within_event_seeds_unique = (
    catalog[
        "within_event_seed"
    ].duplicated().sum()
    == 0
)


catalog_is_sorted = bool(
    catalog[
        "continuous_event_time"
    ].is_monotonic_increasing
)


sampled_count_matches_catalog = (
    total_events
    == len(
        catalog
    )
)


annual_counts_match_catalog = (
    events_per_year[
        "event_count"
    ].sum()
    == len(
        catalog
    )
)


rupture_counts_from_catalog = (
    catalog
    .groupby(
        "rupture_id"
    )
    .size()
    .rename(
        "catalog_occurrences"
    )
)


rupture_count_check = (
    rupture_count_table
    .set_index(
        "rupture_id"
    )[
        "sampled_occurrences"
    ]
    .reindex(
        rupture_count_table[
            "rupture_id"
        ]
    )
    .fillna(0)
    .astype(np.int64)
)


catalog_count_check = (
    rupture_counts_from_catalog
    .reindex(
        rupture_count_table[
            "rupture_id"
        ],
        fill_value=0,
    )
    .astype(np.int64)
)


per_rupture_counts_match = np.array_equal(
    rupture_count_check.to_numpy(),
    catalog_count_check.to_numpy(),
)


validation = pd.DataFrame(
    [
        {
            "check": "Catalog contains earthquake events",
            "passes": len(catalog) > 0,
        },
        {
            "check": "Sampled event count matches catalog rows",
            "passes": sampled_count_matches_catalog,
        },
        {
            "check": "Annual event counts sum to catalog rows",
            "passes": annual_counts_match_catalog,
        },
        {
            "check": "Per-rupture sampled counts match catalog",
            "passes": per_rupture_counts_match,
        },
        {
            "check": "All event identifiers are unique",
            "passes": event_ids_unique,
        },
        {
            "check": "All event seeds are unique",
            "passes": event_seeds_unique,
        },
        {
            "check": "All between-event seeds are unique",
            "passes": between_event_seeds_unique,
        },
        {
            "check": "All within-event seeds are unique",
            "passes": within_event_seeds_unique,
        },
        {
            "check": "Simulation years are within the catalog",
            "passes": bool(
                catalog[
                    "simulation_year"
                ].between(
                    1,
                    CATALOG_YEARS,
                    inclusive="both",
                ).all()
            ),
        },
        {
            "check": "Within-year event times are in [0, 1)",
            "passes": bool(
                (
                    (
                        catalog[
                            "time_within_year"
                        ]
                        >= 0
                    )
                    & (
                        catalog[
                            "time_within_year"
                        ]
                        < 1
                    )
                ).all()
            ),
        },
        {
            "check": "Catalog is ordered by event time",
            "passes": catalog_is_sorted,
        },
        {
            "check": "All source rupture identifiers are recognized",
            "passes": bool(
                catalog[
                    "rupture_id"
                ].isin(
                    rupture_rates[
                        "rupture_id"
                    ]
                ).all()
            ),
        },
        {
            "check": "Total event count is within five standard deviations",
            "passes": abs(
                total_event_z_score
            ) <= 5.0,
        },
        {
            "check": "Nonempty-year count is within five standard deviations",
            "passes": abs(
                nonempty_year_z_score
            ) <= 5.0,
        },
        {
            "check": "Multi-event-year count is within five standard deviations",
            "passes": abs(
                multi_event_year_z_score
            ) <= 5.0,
        },
        {
            "check": "Source-group event counts are statistically reasonable",
            "passes": bool(
                (
                    group_summary[
                        "event_count_z_score"
                    ].abs()
                    <= 5.0
                ).all()
            ),
        },
    ]
)


validation.to_csv(
    CATALOG_VALIDATION_PATH,
    index=False,
)


print("Catalog-level simulation summary:")
display(summary)

print("\nSource-group simulation summary:")
display(
    group_summary[
        [
            "target_group",
            "annual_rate",
            "expected_events",
            "realized_events",
            "event_count_z_score",
            "nonempty_years",
            "minimum_magnitude",
            "maximum_magnitude",
        ]
    ]
)

print("\nAnnual event-count distribution:")
display(
    events_per_year[
        "event_count"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "events_in_year"
    )
    .rename(
        "number_of_years"
    )
    .reset_index()
)

print("\nCatalog validation:")
display(validation)


if not validation["passes"].all():
    failed_checks = validation.loc[
        ~validation["passes"],
        "check",
    ].tolist()

    raise RuntimeError(
        "Annual event catalog validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed_checks
        )
    )


def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as file_object:
        for block in iter(
            lambda: file_object.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(block)

    return digest.hexdigest()


catalog_metadata = {
    "notebook": "03_generate_annual_event_catalog.ipynb",
    "catalog_name": "baseline_mean_rate_catalog_2m_years",
    "catalog_years": CATALOG_YEARS,
    "master_seed": MASTER_SEED,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "bit_generator": "PCG64DXSM",
    "count_seed_spawn_key": list(
        count_seed.spawn_key
    ),
    "year_seed_spawn_key": list(
        year_seed.spawn_key
    ),
    "time_seed_spawn_key": list(
        time_seed.spawn_key
    ),
    "occurrence_model": (
        "Independent Poisson count for each rupture "
        "using weighted_annual_rate"
    ),
    "event_time_model": (
        "Conditional uniform event times over the "
        "two-million-year simulation interval"
    ),
    "empty_year_storage": (
        "Only nonempty years are written to annual_event_counts.csv"
    ),
    "common_random_number_rule": (
        "between_event_seed and within_event_seed must be reused "
        "for all later dependence cases"
    ),
    "rupture_rows": int(
        len(rupture_rates)
    ),
    "catalog_event_rows": int(
        len(catalog)
    ),
    "nonempty_years": observed_nonempty_years,
    "empty_years": observed_empty_years,
    "multi_event_years": observed_multi_event_years,
    "maximum_events_in_one_year": int(
        events_per_year[
            "event_count"
        ].max()
    ),
    "total_annual_rate": total_annual_rate,
    "expected_event_count": expected_total_events,
    "realized_event_count": total_events,
    "total_event_z_score": total_event_z_score,
    "event_catalog": str(
        EVENT_CATALOG_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "event_catalog_sha256": file_sha256(
        EVENT_CATALOG_PATH
    ),
    "annual_event_counts": str(
        ANNUAL_COUNT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "annual_event_counts_sha256": file_sha256(
        ANNUAL_COUNT_PATH
    ),
    "rupture_occurrence_counts": str(
        RUPTURE_COUNT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "rupture_occurrence_counts_sha256": file_sha256(
        RUPTURE_COUNT_PATH
    ),
    "summary": str(
        CATALOG_SUMMARY_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "validation": str(
        CATALOG_VALIDATION_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "validation_passed": bool(
        validation[
            "passes"
        ].all()
    ),
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


CATALOG_METADATA_PATH.write_text(
    json.dumps(
        catalog_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


print("\nCATALOG GENERATION COMPLETE")

print(f"\nCatalog years:          {CATALOG_YEARS:,}")
print(f"Earthquake events:      {len(catalog):,}")
print(f"Nonempty years:         {observed_nonempty_years:,}")
print(f"Empty years:            {observed_empty_years:,}")
print(f"Multi-event years:      {observed_multi_event_years:,}")
print(
    "Maximum events/year:  "
    f"{events_per_year['event_count'].max():,}"
)
print(
    "Realized annual rate: "
    f"{len(catalog) / CATALOG_YEARS:.14g}"
)
print(
    "Target annual rate:   "
    f"{total_annual_rate:.14g}"
)

print(f"\nEvent catalog:\n  {EVENT_CATALOG_PATH}")
print(f"\nAnnual counts:\n  {ANNUAL_COUNT_PATH}")
print(f"\nRupture counts:\n  {RUPTURE_COUNT_PATH}")
print(f"\nMetadata:\n  {CATALOG_METADATA_PATH}")

print(
    "\nThe common stochastic event catalog is ready "
    "for the ground-motion workflow."
)

Catalog-level simulation summary:


,metric,expected,realized,standard_deviation,z_score
0,Total earthquake occurrences,10446.442915,10630,102.207842,1.795920
1,Nonempty catalog years,10419.208311,10593,101.808292,1.707048
2,Multi-event catalog years,27.187228,36,5.214102,1.690180



Source-group simulation summary:


,target_group,annual_rate,expected_events,realized_events,event_count_z_score,nonempty_years,minimum_magnitude,maximum_magnitude
0,cascadia_interface,0.003307,6613.599912,6680,0.816488,6663,8.00,9.34
1,oregon_intraslab,0.001916,3832.843003,3950,1.892377,3943,6.55,7.95



Annual event-count distribution:


,events_in_year,number_of_years
0,1,10557
1,2,35
2,3,1



Catalog validation:


,check,passes
0,Catalog contains earthquake events,True
1,Sampled event count matches catalog rows,True
2,Annual event counts sum to catalog rows,True
3,Per-rupture sampled counts match catalog,True
4,All event identifiers are unique,True
5,All event seeds are unique,True
6,All between-event seeds are unique,True
7,All within-event seeds are unique,True
8,Simulation years are within the catalog,True
9,"Within-year event times are in [0, 1)",True



CATALOG GENERATION COMPLETE

Catalog years:          2,000,000
Earthquake events:      10,630
Nonempty years:         10,593
Empty years:            1,989,407
Multi-event years:      36
Maximum events/year:  3
Realized annual rate: 0.005315
Target annual rate:   0.0052232214574573

Event catalog:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\annual_event_catalog\annual_event_catalog.csv

Annual counts:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\annual_event_catalog\annual_event_counts.csv

Rupture counts:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\processed\annual_event_catalog\rupture_occurrence_counts.csv

Metadata:
  C:\Users\USER\Documents\GitHub\seismic-correlation-insurance-loss\data\metadata\annual_event_catalog_metadata.json

The common stochastic event catalog is ready for the ground-motion workflow.


In [4]:
from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display


FINAL_VALIDATION_PATH = (
    METADATA_DIR
    / "notebook_3_final_validation.csv"
)

COMPLETION_METADATA_PATH = (
    METADATA_DIR
    / "notebook_3_completion_metadata.json"
)


required_paths = {
    "Catalog configuration": CATALOG_CONFIG_PATH,
    "Catalog metadata": CATALOG_METADATA_PATH,
    "Event catalog": EVENT_CATALOG_PATH,
    "Annual event counts": ANNUAL_COUNT_PATH,
    "Rupture occurrence counts": RUPTURE_COUNT_PATH,
    "Cascadia rupture rates": CASCADIA_RATE_PATH,
    "Oregon rupture rates": OREGON_RATE_PATH,
}


missing_paths = [
    path
    for path in required_paths.values()
    if not Path(path).is_file()
]

if missing_paths:
    raise FileNotFoundError(
        "Required Notebook 3 files are missing:\n"
        + "\n".join(
            f"  {path}"
            for path in missing_paths
        )
    )


config = json.loads(
    CATALOG_CONFIG_PATH.read_text(
        encoding="utf-8"
    )
)

catalog_metadata = json.loads(
    CATALOG_METADATA_PATH.read_text(
        encoding="utf-8"
    )
)


catalog_years = int(
    config["catalog_years"]
)

master_seed = int(
    config["master_seed"]
)


cascadia_rates = pd.read_csv(
    CASCADIA_RATE_PATH,
    low_memory=False,
)

oregon_rates = pd.read_csv(
    OREGON_RATE_PATH,
    low_memory=False,
)

rate_inputs = pd.concat(
    [
        cascadia_rates,
        oregon_rates,
    ],
    ignore_index=True,
)


saved_catalog = pd.read_csv(
    EVENT_CATALOG_PATH,
    low_memory=False,
)

saved_annual_counts = pd.read_csv(
    ANNUAL_COUNT_PATH
)

saved_rupture_counts = pd.read_csv(
    RUPTURE_COUNT_PATH,
    low_memory=False,
)


rates = rate_inputs[
    "weighted_annual_rate"
].to_numpy(
    dtype=float
)


seed_sequence = np.random.SeedSequence(
    master_seed
)

count_seed, year_seed, time_seed = (
    seed_sequence.spawn(3)
)


count_rng = np.random.Generator(
    np.random.PCG64DXSM(
        count_seed
    )
)

year_rng = np.random.Generator(
    np.random.PCG64DXSM(
        year_seed
    )
)

time_rng = np.random.Generator(
    np.random.PCG64DXSM(
        time_seed
    )
)


reproduced_counts = count_rng.poisson(
    rates * catalog_years
).astype(
    np.int64
)


reproduced_event_count = int(
    reproduced_counts.sum()
)


rupture_indices = np.repeat(
    np.arange(
        len(rate_inputs),
        dtype=np.int64,
    ),
    reproduced_counts,
)


occurrence_numbers = np.concatenate(
    [
        np.arange(
            1,
            count + 1,
            dtype=np.int64,
        )
        for count in reproduced_counts
        if count > 0
    ]
)


reproduced_catalog = rate_inputs.iloc[
    rupture_indices
].copy()

reproduced_catalog.reset_index(
    drop=True,
    inplace=True,
)


reproduced_catalog.insert(
    0,
    "rupture_occurrence_number",
    occurrence_numbers,
)


reproduced_catalog.insert(
    0,
    "simulation_year",
    year_rng.integers(
        low=1,
        high=catalog_years + 1,
        size=reproduced_event_count,
        dtype=np.int64,
    ),
)


reproduced_catalog.insert(
    1,
    "time_within_year",
    time_rng.random(
        reproduced_event_count
    ),
)


reproduced_catalog.insert(
    2,
    "continuous_event_time",
    (
        reproduced_catalog[
            "simulation_year"
        ].to_numpy(
            dtype=float
        )
        - 1.0
        + reproduced_catalog[
            "time_within_year"
        ].to_numpy(
            dtype=float
        )
    ),
)


def deterministic_seed(
    label,
    rupture_id,
    occurrence_number,
):
    text = (
        f"{master_seed}|"
        f"{label}|"
        f"{rupture_id}|"
        f"{occurrence_number}"
    )

    digest = hashlib.sha256(
        text.encode(
            "utf-8"
        )
    ).digest()

    value = int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False,
    )

    return value & (
        (1 << 63) - 1
    )


seed_inputs = list(
    zip(
        reproduced_catalog[
            "rupture_id"
        ].astype(str),
        reproduced_catalog[
            "rupture_occurrence_number"
        ].astype(int),
    )
)


reproduced_catalog[
    "event_seed"
] = [
    deterministic_seed(
        "event",
        rupture_id,
        occurrence_number,
    )
    for rupture_id, occurrence_number
    in seed_inputs
]


reproduced_catalog[
    "between_event_seed"
] = [
    deterministic_seed(
        "between_event",
        rupture_id,
        occurrence_number,
    )
    for rupture_id, occurrence_number
    in seed_inputs
]


reproduced_catalog[
    "within_event_seed"
] = [
    deterministic_seed(
        "within_event",
        rupture_id,
        occurrence_number,
    )
    for rupture_id, occurrence_number
    in seed_inputs
]


reproduced_catalog.sort_values(
    [
        "simulation_year",
        "time_within_year",
        "rupture_id",
        "rupture_occurrence_number",
    ],
    kind="stable",
    inplace=True,
)

reproduced_catalog.reset_index(
    drop=True,
    inplace=True,
)


reproduced_catalog.insert(
    0,
    "event_id",
    [
        f"EVT_{event_number:08d}"
        for event_number in range(
            1,
            reproduced_event_count + 1,
        )
    ],
)


reproduced_catalog[
    "event_number_in_year"
] = (
    reproduced_catalog
    .groupby(
        "simulation_year",
        sort=False,
    )
    .cumcount()
    + 1
)


reproduced_annual_counts = (
    reproduced_catalog
    .groupby(
        "simulation_year",
        sort=True,
    )
    .size()
    .rename(
        "event_count"
    )
    .reset_index()
)


reproduced_annual_counts[
    "is_multi_event_year"
] = (
    reproduced_annual_counts[
        "event_count"
    ]
    >= 2
)


reproduced_catalog = reproduced_catalog.merge(
    reproduced_annual_counts[
        [
            "simulation_year",
            "event_count",
        ]
    ],
    on="simulation_year",
    how="left",
    validate="many_to_one",
)


reproduced_catalog.rename(
    columns={
        "event_count": "events_in_year",
    },
    inplace=True,
)


saved_catalog = saved_catalog.sort_values(
    "event_id"
).reset_index(
    drop=True
)

reproduced_catalog = (
    reproduced_catalog
    .sort_values(
        "event_id"
    )
    .reset_index(
        drop=True
    )
)


integer_columns = [
    "simulation_year",
    "event_number_in_year",
    "events_in_year",
    "rupture_occurrence_number",
    "event_seed",
    "between_event_seed",
    "within_event_seed",
]


string_columns = [
    "event_id",
    "rupture_id",
    "target_group",
]


# float_columns = [
#     "time_within_year",
#     "continuous_event_time",
# ]


integer_columns_match = all(
    np.array_equal(
        saved_catalog[
            column
        ].to_numpy(
            dtype=np.int64
        ),
        reproduced_catalog[
            column
        ].to_numpy(
            dtype=np.int64
        ),
    )
    for column in integer_columns
)


string_columns_match = all(
    np.array_equal(
        saved_catalog[
            column
        ].astype(str).to_numpy(),
        reproduced_catalog[
            column
        ].astype(str).to_numpy(),
    )
    for column in string_columns
)


# float_columns_match = all(
#     np.allclose(
#         saved_catalog[
#             column
#         ].to_numpy(
#             dtype=float
#         ),
#         reproduced_catalog[
#             column
#         ].to_numpy(
#             dtype=float
#         ),
#         rtol=0.0,
#         atol=1e-12,
#         equal_nan=False,
#     )
#     for column in float_columns
# )
saved_time_within_year = saved_catalog[
    "time_within_year"
].to_numpy(
    dtype=float
)

reproduced_time_within_year = reproduced_catalog[
    "time_within_year"
].to_numpy(
    dtype=float
)


saved_continuous_event_time = saved_catalog[
    "continuous_event_time"
].to_numpy(
    dtype=float
)

reproduced_continuous_event_time = reproduced_catalog[
    "continuous_event_time"
].to_numpy(
    dtype=float
)


time_within_year_tolerance = (
    8.0
    * np.spacing(
        1.0
    )
)

continuous_event_time_tolerance = (
    8.0
    * np.spacing(
        float(
            catalog_years
        )
    )
)


time_within_year_max_abs_difference = float(
    np.max(
        np.abs(
            saved_time_within_year
            - reproduced_time_within_year
        )
    )
)


continuous_event_time_max_abs_difference = float(
    np.max(
        np.abs(
            saved_continuous_event_time
            - reproduced_continuous_event_time
        )
    )
)


time_within_year_matches = bool(
    np.allclose(
        saved_time_within_year,
        reproduced_time_within_year,
        rtol=0.0,
        atol=time_within_year_tolerance,
        equal_nan=False,
    )
)


continuous_event_time_matches = bool(
    np.allclose(
        saved_continuous_event_time,
        reproduced_continuous_event_time,
        rtol=0.0,
        atol=continuous_event_time_tolerance,
        equal_nan=False,
    )
)


expected_saved_continuous_time = (
    saved_catalog[
        "simulation_year"
    ].to_numpy(
        dtype=float
    )
    - 1.0
    + saved_time_within_year
)


saved_continuous_time_is_consistent = bool(
    np.allclose(
        saved_continuous_event_time,
        expected_saved_continuous_time,
        rtol=0.0,
        atol=continuous_event_time_tolerance,
        equal_nan=False,
    )
)


float_columns_match = bool(
    time_within_year_matches
    and continuous_event_time_matches
)

saved_count_table = (
    saved_rupture_counts
    .set_index(
        "rupture_id"
    )[
        "sampled_occurrences"
    ]
    .reindex(
        rate_inputs[
            "rupture_id"
        ]
    )
    .to_numpy(
        dtype=np.int64
    )
)


rupture_counts_match = np.array_equal(
    saved_count_table,
    reproduced_counts,
)


saved_annual_counts_check = (
    saved_annual_counts
    .sort_values(
        "simulation_year"
    )
    .reset_index(
        drop=True
    )
)


reproduced_annual_counts_check = (
    reproduced_annual_counts
    .sort_values(
        "simulation_year"
    )
    .reset_index(
        drop=True
    )
)


annual_years_match = np.array_equal(
    saved_annual_counts_check[
        "simulation_year"
    ].to_numpy(
        dtype=np.int64
    ),
    reproduced_annual_counts_check[
        "simulation_year"
    ].to_numpy(
        dtype=np.int64
    ),
)


annual_event_counts_match = np.array_equal(
    saved_annual_counts_check[
        "event_count"
    ].to_numpy(
        dtype=np.int64
    ),
    reproduced_annual_counts_check[
        "event_count"
    ].to_numpy(
        dtype=np.int64
    ),
)


def file_sha256(path):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as file_object:
        for block in iter(
            lambda: file_object.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(
                block
            )

    return digest.hexdigest()


catalog_hash_matches = (
    file_sha256(
        EVENT_CATALOG_PATH
    )
    == catalog_metadata[
        "event_catalog_sha256"
    ]
)


annual_count_hash_matches = (
    file_sha256(
        ANNUAL_COUNT_PATH
    )
    == catalog_metadata[
        "annual_event_counts_sha256"
    ]
)


rupture_count_hash_matches = (
    file_sha256(
        RUPTURE_COUNT_PATH
    )
    == catalog_metadata[
        "rupture_occurrence_counts_sha256"
    ]
)


validation = pd.DataFrame(
    [
        {
            "check": "Configuration and metadata catalog years match",
            "passes": (
                catalog_years
                == int(
                    catalog_metadata[
                        "catalog_years"
                    ]
                )
            ),
        },
        {
            "check": "Configuration and metadata master seeds match",
            "passes": (
                master_seed
                == int(
                    catalog_metadata[
                        "master_seed"
                    ]
                )
            ),
        },
        {
            "check": "Reproduced event count matches saved catalog",
            "passes": (
                reproduced_event_count
                == len(
                    saved_catalog
                )
            ),
        },
        {
            "check": "Per-rupture Poisson counts reproduce exactly",
            "passes": rupture_counts_match,
        },
        {
            "check": "Event identifiers and rupture assignments reproduce",
            "passes": string_columns_match,
        },
        {
            "check": "Event years and deterministic seeds reproduce",
            "passes": integer_columns_match,
        },
        
        {
            "check": "Within-year event times reproduce",
            "passes": time_within_year_matches,
        },
        {
            "check": (
                "Continuous event times reproduce "
                "within float64 CSV precision"
            ),
            "passes": continuous_event_time_matches,
        },
        {
            "check": (
                "Saved continuous event times are "
                "internally consistent"
            ),
            "passes": saved_continuous_time_is_consistent,
        },
        {
            "check": "Nonempty catalog years reproduce",
            "passes": annual_years_match,
        },
        {
            "check": "Annual event counts reproduce",
            "passes": annual_event_counts_match,
        },
        {
            "check": "Event catalog checksum matches metadata",
            "passes": catalog_hash_matches,
        },
        {
            "check": "Annual-count checksum matches metadata",
            "passes": annual_count_hash_matches,
        },
        {
            "check": "Rupture-count checksum matches metadata",
            "passes": rupture_count_hash_matches,
        },
        {
            "check": "Event identifiers remain unique",
            "passes": (
                saved_catalog[
                    "event_id"
                ].duplicated().sum()
                == 0
            ),
        },
        {
            "check": "Between-event seeds remain unique",
            "passes": (
                saved_catalog[
                    "between_event_seed"
                ].duplicated().sum()
                == 0
            ),
        },
        {
            "check": "Within-event seeds remain unique",
            "passes": (
                saved_catalog[
                    "within_event_seed"
                ].duplicated().sum()
                == 0
            ),
        },
    ]
)


validation.to_csv(
    FINAL_VALIDATION_PATH,
    index=False,
)
print("Floating-point reproduction diagnostics:")

print(
    "Maximum time-within-year difference:     "
    f"{time_within_year_max_abs_difference:.3e}"
)

print(
    "Allowed time-within-year tolerance:      "
    f"{time_within_year_tolerance:.3e}"
)

print(
    "Maximum continuous-time difference:      "
    f"{continuous_event_time_max_abs_difference:.3e}"
)

print(
    "Allowed continuous-time tolerance:       "
    f"{continuous_event_time_tolerance:.3e}"
)

print()

print("Final Notebook 3 validation:")
display(validation)


if not validation[
    "passes"
].all():
    failed_checks = validation.loc[
        ~validation[
            "passes"
        ],
        "check",
    ].tolist()

    raise RuntimeError(
        "Notebook 3 final validation failed:\n"
        + "\n".join(
            f"  {check}"
            for check in failed_checks
        )
    )


completion_metadata = {
    "notebook": (
        "03_generate_annual_event_catalog.ipynb"
    ),
    "catalog_years": catalog_years,
    "master_seed": master_seed,
    "bit_generator": "PCG64DXSM",
    "rupture_rows": int(
        len(
            rate_inputs
        )
    ),
    "earthquake_events": int(
        len(
            saved_catalog
        )
    ),
    "nonempty_years": int(
        len(
            saved_annual_counts
        )
    ),
    "empty_years": int(
        catalog_years
        - len(
            saved_annual_counts
        )
    ),
    "multi_event_years": int(
        (
            saved_annual_counts[
                "event_count"
            ]
            >= 2
        ).sum()
    ),
    "maximum_events_in_one_year": int(
        saved_annual_counts[
            "event_count"
        ].max()
    ),
    "target_annual_rate": float(
        rate_inputs[
            "weighted_annual_rate"
        ].sum()
    ),
    "realized_annual_rate": float(
        len(
            saved_catalog
        )
        / catalog_years
    ),
    "event_catalog": str(
        EVENT_CATALOG_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "event_catalog_sha256": file_sha256(
        EVENT_CATALOG_PATH
    ),
    "annual_event_counts": str(
        ANNUAL_COUNT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "rupture_occurrence_counts": str(
        RUPTURE_COUNT_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "final_validation": str(
        FINAL_VALIDATION_PATH.relative_to(
            PROJECT_ROOT
        )
    ),
    "common_catalog_requirement": (
        "This exact event catalog must be reused for "
        "all later dependence cases."
    ),
    "validation_passed": bool(
        validation[
            "passes"
        ].all()
    ),
    "completed_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}


COMPLETION_METADATA_PATH.write_text(
    json.dumps(
        completion_metadata,
        indent=2,
    ),
    encoding="utf-8",
)


print("\nNOTEBOOK 3 VALIDATION COMPLETE")

print(f"\nCatalog years:          {catalog_years:,}")
print(f"Earthquake events:      {len(saved_catalog):,}")
print(
    "Nonempty years:         "
    f"{len(saved_annual_counts):,}"
)
print(
    "Multi-event years:      "
    f"{(saved_annual_counts['event_count'] >= 2).sum():,}"
)
print(
    "Maximum events/year:    "
    f"{saved_annual_counts['event_count'].max():,}"
)
print(
    "Target annual rate:     "
    f"{rate_inputs['weighted_annual_rate'].sum():.14g}"
)
print(
    "Realized annual rate:   "
    f"{len(saved_catalog) / catalog_years:.14g}"
)

print(
    "\nThe saved catalog was reproduced exactly from "
    "the master seed and validated input rates."
)

print(
    "\nNext notebook:"
    "\n04_generate_ground_motion_fields.ipynb"
)

Floating-point reproduction diagnostics:
Maximum time-within-year difference:     1.110e-16
Allowed time-within-year tolerance:      1.776e-15
Maximum continuous-time difference:      4.657e-10
Allowed continuous-time tolerance:       1.863e-09

Final Notebook 3 validation:


,check,passes
0,Configuration and metadata catalog years match,True
1,Configuration and metadata master seeds match,True
2,Reproduced event count matches saved catalog,True
3,Per-rupture Poisson counts reproduce exactly,True
4,Event identifiers and rupture assignments repr...,True
5,Event years and deterministic seeds reproduce,True
6,Within-year event times reproduce,True
7,Continuous event times reproduce within float6...,True
8,Saved continuous event times are internally co...,True
9,Nonempty catalog years reproduce,True



NOTEBOOK 3 VALIDATION COMPLETE

Catalog years:          2,000,000
Earthquake events:      10,630
Nonempty years:         10,593
Multi-event years:      36
Maximum events/year:    3
Target annual rate:     0.0052232214574573
Realized annual rate:   0.005315

The saved catalog was reproduced exactly from the master seed and validated input rates.

Next notebook:
04_generate_ground_motion_fields.ipynb
